# From circuit to chip: transpilation
**Quantum Computing · Week 6 (Noise and real hardware) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qc/w6d4.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

A real chip only offers a few **native gates**, and only some pairs of its qubits can talk. The
**transpiler** rewrites any circuit to fit: it translates gates, places the qubits, adds SWAPs, and tidies up.

In [ ]:
import itertools
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import CouplingMap

h_only = QuantumCircuit(1)
h_only.h(0)
native = transpile(h_only, basis_gates=["rz", "sx", "x", "cx"])
print(native.draw("text"))
print("same gate (up to a global phase):", Operator(h_only).equiv(Operator(native)))

On IBM chips, $R_Z$ is "virtual": it is done by shifting the phase of later pulses, so it costs no time
and adds no error. Every other gate is built from $\sqrt{X}$, $R_Z$ and a two-qubit gate.

## A chip with a coupling map
`GenericBackendV2` makes a pretend chip. Ours has 5 qubits in a T shape: 0–1–2 across, and 1–3–4 down.

In [ ]:
tee = CouplingMap([[0, 1], [1, 2], [1, 3], [3, 4]])
tee.make_symmetric()
chip = GenericBackendV2(num_qubits=5, coupling_map=tee, basis_gates=["cx", "rz", "sx", "x"], seed=1)

star = QuantumCircuit(5)                       # a GHZ state made with CNOTs from qubit 0 to everyone
star.h(0)
for target in range(1, 5):
    star.cx(0, target)

for level in [0, 3]:
    routed = transpile(star, chip, optimization_level=level, seed_transpiler=1)
    print(f"optimization level {level}: {routed.count_ops().get('cx', 0)} CNOTs (the circuit has 4), depth {routed.depth()}")
transpile(star, chip, optimization_level=3, seed_transpiler=1).draw("mpl", style="clifford", fold=-1, idle_wires=False)

Qubit 0 can't reach qubits 2, 3 and 4 directly, so the transpiler moves qubits around with SWAPs, each
made of three CNOTs. Higher optimization levels search for a better placement and cancel gates.

## Why it matters: every CNOT adds noise
Run the original and the routed circuits on a simulator with a 1% error on every CNOT.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ["cx"])
sim = AerSimulator(noise_model=noise)

measured = star.copy()
measured.measure_all()
for name, circuit in [("all-to-all chip (no SWAPs)", transpile(measured, sim, optimization_level=0)),
                      ("T-shaped chip, level 0", transpile(measured, chip, optimization_level=0, seed_transpiler=1)),
                      ("T-shaped chip, level 3", transpile(measured, chip, optimization_level=3, seed_transpiler=1))]:
    counts = sim.run(circuit, shots=8000, seed_simulator=2).result().get_counts()
    good = (counts.get("00000", 0) + counts.get("11111", 0)) / 8000
    print(f"{name:28s}: {circuit.count_ops().get('cx', 0):2d} CNOTs, correct outcomes {good:.3f}")

## Your turn: place the qubits yourself
Take a **chain** circuit, CNOTs 0→1, 1→2, 2→3, 3→4, on the T-shaped chip. Try every one of the
5! = 120 initial layouts (`initial_layout=list(perm)` tells the transpiler where each qubit starts)
and find the fewest CNOTs. Is the transpiler's own choice at level 3 as good?

In [ ]:
chain = QuantumCircuit(5)
chain.h(0)
for q in range(4):
    chain.cx(q, q + 1)

counts = {}
for perm in itertools.permutations(range(5)):
    counts[perm] = 0     # TODO: transpile(chain, chip, initial_layout=list(perm), optimization_level=1,
                         #       seed_transpiler=1), then .count_ops().get("cx", 0)
print("fewest CNOTs:", min(counts.values()))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
chain = QuantumCircuit(5)
chain.h(0)
for q in range(4):
    chain.cx(q, q + 1)

counts = {}
for perm in itertools.permutations(range(5)):
    routed = transpile(chain, chip, initial_layout=list(perm), optimization_level=1, seed_transpiler=1)
    counts[perm] = routed.count_ops().get("cx", 0)
best = min(counts.values())
print(f"fewest CNOTs over all 120 layouts: {best} (the chain has 4); worst: {max(counts.values())}")
print("best layouts:", [p for p, c in counts.items() if c == best])
auto = transpile(chain, chip, optimization_level=3, seed_transpiler=1)
print(f"the transpiler's own choice at level 3: {auto.count_ops().get('cx', 0)} CNOTs")
print("A chain of 5 doesn't fit on a T without a detour, and smart placement matters a lot. Automatic layout")
print("is a heuristic: good, not always the best.")